In [2]:
# Data preprocessing for all datasets
# This notebook converts raw UCI data files to unified CSV format.
# The unified format is:
# - CSV file with columns: id, feature_1, feature_2, ..., target
# - target values are integers starting from 0
# - dataset_info.json with metadata

# 0.Setup
import pandas as pd
import numpy as np
from pathlib import Path
import json
import os

PROJECT_ROOT = Path.cwd().parent
DATA_DIR = PROJECT_ROOT / 'data'
DATASET_FOLDERS = {
    'iris': '1.Numerical_iris',
    'parkinsons': "2.Numerical_parkinsons",
    'hepatitis': "3.Mixed_hepatitis",
    'acute_inflammations': "4.Mixed_acute_inflammations",
    'zoo': '5.Categorical_zoo',
    'hayes_roth': "6.Categorical_hayes_roth",
    #'lenses': "6.Categorical_lenses",
}

print("="*60)
print("Data Preprocessing Started")
print("="*60)



Data Preprocessing Started


In [27]:
# 1. Iris dataset (Numerical, 150 samples, 4 features, 3 classes)
# Original: sepal_length, sepal_width, petal_length, petal_width, class
def preprocess_iris():
    folder = DATA_DIR / DATASET_FOLDERS['iris']
    raw_path = folder / 'iris.data'

    column_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width', 
                    'target']
    df = pd.read_csv(raw_path, header=None, names=column_names)

    class_mapping = {'Iris-setosa': 0, 'Iris-versicolor': 1, 'Iris-virginica': 2}
    df['target'] = df['target'].map(class_mapping)

    df.insert(0, 'id', range(len(df)))

    df.to_csv(folder / 'iris.csv', index=False)

    info = {
        'name': 'iris',
        'type': 'numerical',
        'samples': len(df),
        'feature': 4,
        'classes': 3,
        'class_names': ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica'],
        'feature_names': ['sepal_length', 'sepal_width', 'petal_length', 'petal_width'],
        'has_header': True,
        'target_column': 'target'        
    }

    with open(folder / 'dataset_info.json', 'w') as f:
        json.dump(info, f, indent=4)

    print(f" Iris: {len(df)} samples, 4 features, 3 classes")
    print(f" Saved to: {folder / 'iris.csv'}")
    return df

preprocess_iris()

 Iris: 150 samples, 4 features, 3 classes
 Saved to: d:\Develop\PyTorch_26Project\data\1.Numerical_iris\iris.csv


,id,sepal_length,sepal_width,petal_length,petal_width,target
0,0,5.1,3.5,1.4,0.2,0
1,1,4.9,3.0,1.4,0.2,0
2,2,4.7,3.2,1.3,0.2,0
3,3,4.6,3.1,1.5,0.2,0
4,4,5.0,3.6,1.4,0.2,0
...,...,...,...,...,...,...
145,145,6.7,3.0,5.2,2.3,2
146,146,6.3,2.5,5.0,1.9,2
147,147,6.5,3.0,5.2,2.0,2
148,148,6.2,3.4,5.4,2.3,2


In [8]:
# 2. Parkinsons Dataset (Numerical, 195 samples, 22 features, 2 classes)
# Original: name,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),
#  MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,MDVP:Shimmer(dB),Shimmer:APQ3,Shimmer:APQ5,
#  MDVP:APQ,Shimmer:DDA,NHR,HNR,status,RPDE,DFA,spread1,spread2,D2,PPE

def preprocess_parkinsons():
    folder = DATA_DIR / DATASET_FOLDERS['parkinsons']
    raw_path = folder / 'parkinsons.data'
    
    # read raw data (header exists)
    df = pd.read_csv(raw_path)
    
    # drop 'name' column (non-features, just ASCII identifier)
    df = df.drop(columns=['name'])
    
    # status is target column（0=healthy, 1=parkinson)
    df = df.rename(columns={'status': 'target'})
    
    # add id 
    df.insert(0, 'id', range(len(df)))
    
    # save
    df.to_csv(folder / 'parkinsons.csv', index=False)
    
    # create info
    feature_names = df.columns[1:-1].tolist()  # exclude id and target columns
    info = {
        'name': 'parkinsons',
        'type': 'numerical',
        'samples': len(df),
        'features': len(feature_names),
        'classes': 2,
        'class_names': ['Healthy', 'Parkinson'],
        'feature_names': feature_names,
        'has_header': True,
        'target_column': 'target',
        'features_to_drop': ['name']
    }
    with open(folder / 'dataset_info.json', 'w') as f:
        json.dump(info, f, indent=4)
    
    print(f" Parkinsons: {len(df)} samples, {len(feature_names)} features, 2 classes")
    return df

preprocess_parkinsons()




 Parkinsons: 195 samples, 22 features, 2 classes


,id,MDVP:Fo(Hz),MDVP:Fhi(Hz),MDVP:Flo(Hz),MDVP:Jitter(%),MDVP:Jitter(Abs),MDVP:RAP,MDVP:PPQ,Jitter:DDP,MDVP:Shimmer,...,Shimmer:DDA,NHR,HNR,target,RPDE,DFA,spread1,spread2,D2,PPE
0,0,119.992,157.302,74.997,0.00784,0.00007,0.00370,0.00554,0.01109,0.04374,...,0.06545,0.02211,21.033,1,0.414783,0.815285,-4.813031,0.266482,2.301442,0.284654
1,1,122.400,148.650,113.819,0.00968,0.00008,0.00465,0.00696,0.01394,0.06134,...,0.09403,0.01929,19.085,1,0.458359,0.819521,-4.075192,0.335590,2.486855,0.368674
2,2,116.682,131.111,111.555,0.01050,0.00009,0.00544,0.00781,0.01633,0.05233,...,0.08270,0.01309,20.651,1,0.429895,0.825288,-4.443179,0.311173,2.342259,0.332634
3,3,116.676,137.871,111.366,0.00997,0.00009,0.00502,0.00698,0.01505,0.05492,...,0.08771,0.01353,20.644,1,0.434969,0.819235,-4.117501,0.334147,2.405554,0.368975
4,4,116.014,141.781,110.655,0.01284,0.00011,0.00655,0.00908,0.01966,0.06425,...,0.10470,0.01767,19.649,1,0.417356,0.823484,-3.747787,0.234513,2.332180,0.410335
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,190,174.188,230.978,94.261,0.00459,0.00003,0.00263,0.00259,0.00790,0.04087,...,0.07008,0.02764,19.517,0,0.448439,0.657899,-6.538586,0.121952,2.657476,0.133050
191,191,209.516,253.017,89.488,0.00564,0.00003,0.00331,0.00292,0.00994,0.02751,...,0.04812,0.01810,19.147,0,0.431674,0.683244,-6.195325,0.129303,2.784312,0.168895
192,192,174.688,240.005,74.287,0.01360,0.00008,0.00624,0.00564,0.01873,0.02308,...,0.03804,0.10715,17.883,0,0.407567,0.655683,-6.787197,0.158453,2.679772,0.131728
193,193,198.764,396.961,74.904,0.00740,0.00004,0.00370,0.00390,0.01109,0.02296,...,0.03794,0.07223,19.020,0,0.451221,0.643956,-6.744577,0.207454,2.138608,0.123306


In [ ]:
# 3. Hepatitis (Mixed, 155 samples, 19 features, 2 classes)
# Original: class, age, sex, steroid, ..., histology
# Missing values (marked as '?') are imputed: mode for categorical, median for numerical
# handle missing values with different strategy
def preprocess_hepatitis():
    folder = DATA_DIR / DATASET_FOLDERS['hepatitis']
    raw_path = folder / 'hepatitis.data'

    column_names = [
        'target', 'age', 'sex', 'steroid', 'antivirals', 'fatigue', 'malaise',
        'anorexia', 'liver_big', 'liver_firm', 'spleen_palpable', 'spiders',
        'ascites', 'varices', 'bilirubin', 'alk_phosphate', 'sgot', 'albumin',
        'protime', 'histology'
    ]

    # read raw data, missing value = ?
    df = pd.read_csv(raw_path, header=None, names=column_names, na_values='?')

    # target column: die->0, live->1
    df['target'] = df['target'].map({1: 0, 2: 1})

    # 1.Binary categorical features: Impute with the mode (most frequent number)
    categorical_cols = ['sex', 'steroid', 'antivirals', 'fatigue', 'malaise',
                   'anorexia', 'liver_big', 'liver_firm', 'spleen_palpable',
                   'spiders', 'ascites', 'varices', 'histology']
    
    # fill missing values first
    for col in categorical_cols:
        mode_series = df[col].mode()
        fill_val = mode_series[0] 
        df[col] = df[col].fillna(fill_val)
        # then mapping
        # df[col] = df[col].map({1: 0, 2: 1})

    # 2. Numerical features: Impute with the median
    numerical_cols = ['age', 'bilirubin', 'alk_phosphate', 'sgot', 'albumin', 'protime']   
    for col in numerical_cols:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)


    # 3. One-Hot encoding, sex_1, sex_2 (0, 1)
    df = pd.get_dummies(df, columns=categorical_cols, dtype=int)
    # df = df.astype(float)

    # # 4. Numerical normalization [0, 1]
    # for col in df.columns:
    #     if col not in ['id', 'target']:
    #         col_min = df[col].min()
    #         col_max = df[col].max()
    #         # avoid divide 0
    #         if col_max > col_min:
    #             df[col] = (df[col] - col_min) / (col_max - col_min)
    #         else:
    #             df[col] = 0.0
    
    # add id
    df.insert(0, 'id', range(len(df)))

    df.to_csv(folder / 'hepatitis.csv', index=False)

    info = {
        'name': 'hepatitis',
        'type': 'mixed',
        'samples': len(df),
        'features': len(df.columns) - 2,
        'classes': 2,
        'class_names': ['Die', 'Live'],
        'feature_names': [c for c in df.columns if c not in ['id', 'target']],
        'has_header': True,
        'target_column': 'target'
    }

    with open(folder / 'dataset_info.json', 'w') as f:
        json.dump(info, f, indent=4)
    
    print(f" Hepatitis: {len(df)} samples, 19 features, 2 classes")
    return df

preprocess_hepatitis()

In [37]:
# 4. Acute inflammations (Mixed, 120 samples, 6 features, 2 classes)
def preprocess_acute_inflammations():
    folder = DATA_DIR / DATASET_FOLDERS['acute_inflammations']
    raw_path = folder / 'diagnosis.data'

    column_names = [
        'temperature', 'nausea', 'lumbar_pain', 'urine_pushing',
        'micturition_pain', 'burning_urethra', 'inflammation', 'nephritis'
    ]    

    # seperate by tab
    df = pd.read_csv(raw_path, header=None, names=column_names, 
                     sep='\t', encoding='utf-16', decimal=',')

    # binary transfer
    binary_cols = ['nausea', 'lumbar_pain', 'urine_pushing',
                   'micturition_pain', 'burning_urethra']
    
    for col in binary_cols:
        df[col] = df[col].map({'no': 0, 'yes': 1})

    # choose inflammation class only
    df['target'] = df['inflammation'].map({'no': 0, 'yes': 1})

    df = df.drop(columns=['inflammation', 'nephritis'])

    # One-hot encoding for categorical features
    categorical_cols = ['temperature', 'nausea', 'lumbar_pain', 
                     'urine_pushing', 'micturition_pain', 'burning_urethra']  
    df = pd.get_dummies(df, columns=categorical_cols, dtype=int)

    df.insert(0, 'id', range(len(df)))

    df.to_csv(folder / 'acute_inflammations.csv', index=False)

    info = {
        'name': 'acute_inflammations',
        'type': 'mixed',
        'samples': len(df),
        'features': len(df.columns) - 2,
        'classes': 2,
        'class_names': ['No Inflammation', 'Inflammation'],
        'feature_names':  [c for c in df.columns if c not in ['id', 'target']],
        'has_header': True,
        'target_column': 'target'
    }   

    with open(folder / 'dataset_info.json', 'w') as f:
        json.dump(info, f, indent=4)
    
    print(f" Acute inflammations: {len(df)} samples, {len(df.columns) - 2} features, 2 classes")
    return df

preprocess_acute_inflammations()

 Acute inflammations: 120 samples, 54 features, 2 classes


,id,target,temperature_35.5,temperature_35.9,temperature_36.0,temperature_36.2,temperature_36.3,temperature_36.6,temperature_36.7,temperature_36.8,...,nausea_0,nausea_1,lumbar_pain_0,lumbar_pain_1,urine_pushing_0,urine_pushing_1,micturition_pain_0,micturition_pain_1,burning_urethra_0,burning_urethra_1
0,0,0,1,0,0,0,0,0,0,0,...,1,0,0,1,1,0,1,0,1,0
1,1,1,0,1,0,0,0,0,0,0,...,1,0,1,0,0,1,0,1,0,1
2,2,0,0,1,0,0,0,0,0,0,...,1,0,0,1,1,0,1,0,1,0
3,3,1,0,0,1,0,0,0,0,0,...,1,0,1,0,0,1,0,1,0,1
4,4,0,0,0,1,0,0,0,0,0,...,1,0,0,1,1,0,1,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115,115,0,0,0,0,0,0,0,0,0,...,1,0,0,1,0,1,1,0,0,1
116,116,0,0,0,0,0,0,0,0,0,...,1,0,1,0,1,0,1,0,1,0
117,117,0,0,0,0,0,0,0,0,0,...,0,1,0,1,1,0,0,1,1,0
118,118,0,0,0,0,0,0,0,0,0,...,1,0,0,1,0,1,1,0,0,1


In [38]:
# 5.Zoo (Categorical)
def preprocess_zoo():
    folder = DATA_DIR / DATASET_FOLDERS['zoo']
    raw_path = folder / 'zoo.data'

    column_names = [
        'animal_name', 'hair', 'feathers', 'eggs', 'milk', 'airborne', 'aquatic',
        'predator', 'toothed', 'backbone', 'breathes', 'venomous', 'fins', 'legs',
        'tail', 'domestic', 'catsize', 'target'
    ]

    df = pd.read_csv(raw_path, header=None, names=column_names)

    # delete animal_name, identifier, non-feature
    df = df.drop(columns=['animal_name'])

    # target: type(1-7) -> target(0-6)
    df['target'] = df['target'] - 1

    # One-hot encoding: legs (multi-value, no ordinal relationship)
    df = pd.get_dummies(df, columns=['legs'], prefix='legs', dtype=int)

    if df.isna().sum().sum() > 0:
        print(f"  Warning: Found {df.isna().sum().sum()} missing values")
        df = df.dropna()        

    # add id
    df.insert(0, 'id', range(len(df)))
    df.to_csv(folder / 'zoo.csv', index=False)    

    info = {
        'name': 'zoo',
        'type': 'categorical',
        'samples': len(df),
        'features': len(df.columns) - 2,
        'classes': 7,
        'class_names': [
            'Class_1', 'Class_2', 'Class_3', 'Class_4',
            'Class_5', 'Class_6', 'Class_7'
        ],
        'feature_names': [c for c in df.columns if c not in ['id', 'target']],
        'has_header': True,
        'target_column': 'target',
        'categorical_features': [],  
        'numerical_features': ['legs']
    }    

    with open(folder / 'dataset_info.json', 'w') as f:
        json.dump(info, f, indent=4)

    print(f" Zoo: {len(df)} samples, {len(df.columns) - 2} features, 7 classes")
    print(f"  Class distribution: {df['target'].value_counts().sort_index().tolist()}")
    return df

preprocess_zoo()

 Zoo: 101 samples, 21 features, 7 classes
  Class distribution: [41, 20, 5, 13, 4, 8, 10]


,id,hair,feathers,eggs,milk,airborne,aquatic,predator,toothed,backbone,...,tail,domestic,catsize,target,legs_0,legs_2,legs_4,legs_5,legs_6,legs_8
0,0,1,0,0,1,0,0,1,1,1,...,0,0,1,0,0,0,1,0,0,0
1,1,1,0,0,1,0,0,0,1,1,...,1,0,1,0,0,0,1,0,0,0
2,2,0,0,1,0,0,1,1,1,1,...,1,0,0,3,1,0,0,0,0,0
3,3,1,0,0,1,0,0,1,1,1,...,0,0,1,0,0,0,1,0,0,0
4,4,1,0,0,1,0,0,1,1,1,...,1,0,1,0,0,0,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96,96,1,0,0,1,0,0,0,1,1,...,1,0,1,0,0,1,0,0,0,0
97,97,1,0,1,0,1,0,0,0,0,...,0,0,0,5,0,0,0,0,1,0
98,98,1,0,0,1,0,0,1,1,1,...,1,0,1,0,0,0,1,0,0,0
99,99,0,0,1,0,0,0,0,0,0,...,0,0,0,6,1,0,0,0,0,0


In [ ]:
# 6. hayes_roth (Categorical, 160 samples (train 132 /test 28), 4 features, 3 classes)
# Original data comes with separate train and test files; we merge them and re-split via 5-fold CV
def preprocess_hayes_roth():
    folder = DATA_DIR / DATASET_FOLDERS['hayes_roth']

    names_with_name = ['name', 'hobby', 'age', 'educational_level', 'marital_status', 'target']
    names_no_name   = ['hobby', 'age', 'educational_level', 'marital_status', 'target'] 

    # Train file has a 'name' column, test file does not; read with different column lists
    df_train = pd.read_csv(folder / 'hayes-roth.data', header=None, names=names_with_name, sep=',')
    df_test = pd.read_csv(folder / 'hayes-roth.test', header=None, names=names_no_name, sep=',')

    df_train = df_train.drop(columns=['name'])
    # Merge original train(132) + test(28) for unified 5-fold cross-validation
    df = pd.concat([df_train, df_test], ignore_index=True)

    # Integer mapping: original 1-based values to 0-based for sklearn compatibility
    df['hobby'] = df['hobby'].map({1: 0, 2: 1, 3: 2}).astype(int)
    df['age'] = df['age'].map({1: 0, 2: 1, 3: 2, 4: 3}).astype(int)
    df['educational_level'] = df['educational_level'].map({1: 0, 2: 1, 3: 2, 4: 3}).astype(int)
    df['marital_status'] = df['marital_status'].map({1: 0, 2: 1, 3: 2, 4: 3}).astype(int)
    df['target'] = df['target'].map({1: 0, 2: 1, 3: 2}).astype(int)

    # One-hot encoding for all categorical features  
    categorical_cols = ['hobby', 'age', 'educational_level', 'marital_status']  
    df = pd.get_dummies(df, columns=categorical_cols, dtype=int)

    df.insert(0, 'id', range(len(df)))
  
    df.to_csv(folder / 'hayes_roth.csv', index=False)

    info = {
        'name': 'hayes_roth',
        'type': 'Categorical',
        'samples': len(df),
        'features': len(df.columns) - 2,
        'classes': 3,
        'class_names': ['Class_1', 'Class_2', 'Class_Neither'],
        'feature_names': [c for c in df.columns if c not in ['id', 'target']],
        'has_header': True,
        'target_column': 'target'
    }   

    with open(folder / 'dataset_info.json', 'w') as f:
        json.dump(info, f, indent=4)
    
    print(f" Hayes_Roth: {len(df)} samples, {len(df.columns) - 2} features, 3 classes")
    return df

preprocess_hayes_roth()

In [ ]:
# 6. Lenses (Categorical, 24 samples, 4 features, 3 classes)
# DEPRECATED: replaced by Hayes-Roth (lenses is too small for 5-fold CV).
# Code and results preserved for reference but not used in the paper.
# def preprocess_lenses():
#     folder = DATA_DIR / DATASET_FOLDERS['lenses']
#     raw_path = folder / 'lenses.data'

#     column_names = [
#         'id', 'age', 'spectacle_prescription', 'astigmatic', 'tear_production_rate',
#         'target'
#     ]    

#     # seperate by blank
#     df = pd.read_csv(raw_path, header=None, names=column_names, sep='\s+')

#     # seperate mapping
#     df['age'] = df['age'].map({1: 0, 2: 1, 3: 2})
#     df['spectacle_prescription'] = df['spectacle_prescription'].map({1: 0, 2: 1})
#     df['astigmatic'] = df['astigmatic'].map({1: 0, 2: 1})
#     df['tear_production_rate'] = df['tear_production_rate'].map({1: 0, 2: 1})
#     df['target'] = df['target'].map({1: 0, 2: 1, 3: 2})

#     # One-hot encoding for all categorical features  
#     categorical_cols = ['age', 'spectacle_prescription', 'astigmatic', 'tear_production_rate']  
#     df = pd.get_dummies(df, columns=categorical_cols, dtype=int)
  
#     df.to_csv(folder / 'lenses.csv', index=False)

#     info = {
#         'name': 'lenses',
#         'type': 'Categorical',
#         'samples': len(df),
#         'features': len(df.columns) - 2,
#         'classes': 3,
#         'class_names': ['hard', 'soft', 'no contact lenses'],
#         'feature_names': [c for c in df.columns if c not in ['id', 'target']],
#         'has_header': True,
#         'target_column': 'target'
#     }   

#     with open(folder / 'dataset_info.json', 'w') as f:
#         json.dump(info, f, indent=4)
    
#     print(f" Lenses: {len(df)} samples, {len(df.columns) - 2} features, 3 classes")
#     return df

# preprocess_lenses()